# Stage 3 — Multi-Model Evaluation
**Models:** Llama 3.3 70B · Llama 3.1 70B · Phi-4 14B (via Groq)

**Tasks:**
- Run batch pipeline on all 3 models
- Score physical metrics per project per model
- Arduino CLI compilation checks
- Intra-family comparison (Llama 3.1 vs 3.3)
- Ablation: single-agent vs single-agent+RAG vs multi-agent+RAG

In [ ]:
!pip install fuzzywuzzy python-Levenshtein

## Imports

In [ ]:
import pandas as pd
import json
import os
import time
import subprocess
import re
from tqdm import tqdm
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()
print('All imports OK')

##  Paths & Config

In [ ]:
import os
os.chdir("../.")

In [ ]:


DICT_PATH   = './data/components_dictionary.xlsx'
GT_PATH     = './outputs/benchmark_outputs/ground_truth_final.json'
STAGE2_DIR  = './outputs/generated_code/stage2_outputs'
OUTPUT_DIR  = './outputs/generated_code/stage3_outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

# ── Phantom IDs to suppress from predictions (never from GT) ────────────
# 87 = Arduino Uno (board, not a component); 59 = generic LED (too ambiguous)
PHANTOM_IDS = {87, 59}

def normalise_lib_name(name):
    """Strip .h suffix, lowercase, and strip whitespace for consistent library comparison."""
    return str(name).strip().lower().removesuffix('.h')
print(f'PHANTOM_IDS defined: {PHANTOM_IDS}')

# mistral requires explicit JSON mode enforcement (outputs prose by default)
MISTRAL_JSON_MODE = True
print(f'MISTRAL_JSON_MODE: {MISTRAL_JSON_MODE}')

In [ ]:
# import os
# os.chdir("text-to-arduino")

GT_PATH = os.path.expanduser(
    "~/Desktop/text-to-arduino/outputs/benchmark_outputs/ground_truth_final.json"
)

## Load Ground Truth & Dictionary

In [ ]:
import pandas as pd

# Load descriptions from Excel
meta_df = pd.read_excel(DICT_PATH.replace('components_dictionary.xlsx', 'projects_metadata.xlsx'))

# Build lookup: project_id → description
# Column has trailing space — strip it
meta_df.columns = [c.strip() for c in meta_df.columns]
desc_lookup = dict(zip(meta_df['Project_ID'].astype(int), meta_df['Description'].fillna('')))

# Load ground truth
with open(GT_PATH) as f:
    ground_truth = json.load(f)

in_scope = [r for r in ground_truth if not r.get('out_of_scope', False)]

# Merge descriptions in
for r in in_scope:
    pid = int(r.get('project_id', 0))
    r['description'] = desc_lookup.get(pid, '').strip()

# Verify
has_desc = sum(1 for r in in_scope if r.get('description', '').strip())
print(f'Total projects: {len(ground_truth)}')
print(f'In-scope projects: {len(in_scope)}')
print(f'Projects with descriptions: {has_desc} / {len(in_scope)}')

# Spot check
sample = next(r for r in in_scope if r.get('description'))
print(f'\nSample — Project {sample["project_id"]}: {sample["description"][:120]}')

In [ ]:
with open(GT_PATH) as f:
    ground_truth = json.load(f)

in_scope = [r for r in ground_truth if not r.get('out_of_scope', False)]

# ── Fix description key — Excel column has trailing space ──────
for r in in_scope:
    desc = (
        r.get('Description ') or   # trailing space (from Excel)
        r.get('Description') or    # no space
        r.get('description ') or   # lowercase with space
        r.get('description') or    # lowercase no space
        ''
    ).strip()
    r['description'] = desc        # normalise to clean lowercase key

# Verify
has_desc = sum(1 for r in in_scope if r.get('description', '').strip())
print(f'Total projects: {len(ground_truth)}')
print(f'In-scope projects: {len(in_scope)}')
print(f'Projects with descriptions: {has_desc} / {len(in_scope)}')

## HardwareDictionary (from Stage 2)

In [ ]:
class HardwareDictionary:
    def __init__(self, xlsx_path):
        df = pd.read_excel(xlsx_path)
        self.db = {}
        self.alias_index = {}

        for _, row in df.iterrows():
            cid = int(row['Component_Id'])
            lib_raw = str(row['Associated Libraries'])
            libs = [] if lib_raw.strip() in ['None', 'nan', ''] else [l.strip() for l in lib_raw.split(',')]
            fn_raw = str(row['Function Names'])
            fns = [] if fn_raw.strip() in ['None', 'nan', ''] else [f.strip() for f in fn_raw.split(',')]
            ver_raw = str(row['Common versions'])
            versions = [] if ver_raw.strip() in ['None', 'nan', ''] else [v.strip() for v in ver_raw.split(',')]

            pin_raw = row['Pin Description']
            try:
                pins = json.loads(pin_raw) if isinstance(pin_raw, str) else []
            except Exception:
                pins = []

            self.db[cid] = {
                'id':          cid,
                'name':        str(row['Component name']).strip(),
                'type':        str(row['Component Type']).strip(),
                'category':    str(row['Category']).strip(),
                'subcategory': str(row['Sub-category']).strip(),
                'versions':    versions,
                'libraries':   libs,
                'pin_type':    str(row['Pin Type']).strip(),
                'pins':        pins,
                'use_cases':   str(row['Typical Use Cases']).strip(),
                'functions':   fns,
                'notes':       str(row['Notes']).strip(),
            }
            for variant in [row['Component name']] + versions:
                key = str(variant).strip().lower()
                if key and key not in ['nan', 'none']:
                    self.alias_index[key] = cid

        self._supplement_aliases()

    def _supplement_aliases(self):
        """
        Extended alias table covering the most-missed component categories.
        """
        ALIAS_SUPPLEMENTS = {
            68:  ['lcd i2c', 'i2c lcd', '16x2 i2c', '1602 i2c', 'lcd display i2c',
                  'character lcd i2c', 'i2c display', 'pcf8574 lcd', '2004 lcd', 'lcd2004 i2c'],
            93:  ['esp8266', 'wifi module', 'esp-01', 'esp01', 'wireless module',
                  'wi-fi module', 'nodemcu', 'wifi serial', 'uart wifi', 'esp module'],
            119: ['l298n', 'motor driver', 'dual motor driver', 'h-bridge', 'dc motor driver',
                  'l298 motor driver', 'motor control module'],
            118: ['stepper driver', 'a4988', 'drv8825', 'stepper motor driver', 'step driver'],
            82:  ['encoder', 'rotary encoder', 'wheel encoder', 'motor encoder', 'ky-040'],
        }
        for cid, aliases in ALIAS_SUPPLEMENTS.items():
            if cid in self.db:
                for alias in aliases:
                    k = alias.lower().strip()
                    if k not in self.alias_index:
                        self.alias_index[k] = cid

    def lookup_by_id(self, cid):
        return self.db.get(int(cid))

    def lookup_by_name(self, name):
        cid = self.alias_index.get(str(name).strip().lower())
        return self.db[cid] if cid else None

    def get_canonical_names_list(self):
        return [{'id': e['id'], 'name': e['name']} for e in self.db.values()]

    def get_prompt_context(self, cid, compact=False):
        """
        Returns a structured text block for one component.
        """
        entry = self.lookup_by_id(cid)
        if not entry:
            return f'[WARNING: Component ID {cid} not found]'

        lines = []
        lines.append(f"## {entry['name']}  (ID {entry['id']})")
        lines.append(f"Type: {entry['type']} | Category: {entry['category']} / {entry['subcategory']}")

        if entry['libraries']:
            libs_fmt = ', '.join(f'<{l}>' if not l.endswith('.h') else f'<{l}>'
                                 for l in entry['libraries'])
            lines.append(f"Required #include: {libs_fmt}")
        else:
            lines.append("Required #include: none (uses built-in Arduino functions only)")

        if entry['functions']:
            lines.append(f"Key Arduino functions: {', '.join(entry['functions'])}")

        if entry['pins']:
            lines.append("Pin wiring:")
            for pin in entry['pins']:
                name = pin.get('name', '')
                kind = pin.get('kind', '')
                direction = pin.get('direction', '')
                required = pin.get('required', True)
                bus_role = pin.get('bus_role') or ''
                if kind == 'power':
                    lines.append(f"  {name} → 5V (power)")
                elif kind == 'ground':
                    lines.append(f"  {name} → GND")
                elif 'SDA' in bus_role or name.upper() == 'SDA':
                    lines.append(f"  {name} → A4 (SDA — I2C)")
                elif 'SCL' in bus_role or name.upper() == 'SCL':
                    lines.append(f"  {name} → A5 (SCL — I2C)")
                elif 'MOSI' in bus_role: lines.append(f"  {name} → pin 11 (SPI MOSI)")
                elif 'MISO' in bus_role: lines.append(f"  {name} → pin 12 (SPI MISO)")
                elif 'SCK'  in bus_role: lines.append(f"  {name} → pin 13 (SPI SCK)")
                elif 'SS'   in bus_role: lines.append(f"  {name} → pin 10 (SPI SS)")
                elif kind == 'digital' and direction == 'input':
                    lines.append(f"  {name} → any digital pin  [pinMode OUTPUT from Arduino]" + ("" if required else "  [optional]"))
                elif kind == 'digital' and direction == 'output':
                    lines.append(f"  {name} → any digital pin  [pinMode INPUT to Arduino]" + ("" if required else "  [optional]"))
                elif kind == 'analog':
                    lines.append(f"  {name} → analog pin (A0–A5)  [analogRead()]" + ("" if required else "  [optional]"))
                elif kind == 'pwm':
                    lines.append(f"  {name} → PWM pin (3/5/6/9/10/11 only)")
                else:
                    lines.append(f"  {name} → {kind or 'see datasheet'}")

        if not compact and entry['notes'] and entry['notes'] not in ('nan', 'None', ''):
            lines.append(f"Notes: {entry['notes']}")

        return '\n'.join(lines)

    def build_rag_context(self, component_ids, compact=False, max_chars=None):
        """
        Build the full RAG context string for a list of component IDs.
        """
        if not component_ids:
            return "(no components identified — use standard Arduino patterns)"

        blocks = [self.get_prompt_context(cid, compact=compact)
                  for cid in component_ids]

        if max_chars is None:
            return '\n\n---\n\n'.join(blocks)

        kept = []
        total = 0
        sep = '\n\n---\n\n'
        for i, block in enumerate(blocks):
            cost = len(block) + (len(sep) if kept else 0)
            if total + cost > max_chars and kept:
                dropped = len(blocks) - i
                kept.append(f"[{dropped} component(s) omitted — context limit reached]")
                break
            kept.append(block)
            total += cost
        return sep.join(kept)

dictionary = HardwareDictionary(DICT_PATH)
print(f'HardwareDictionary loaded: {len(dictionary.db)} components, {len(dictionary.alias_index)} aliases (inc. supplements)')

##  LLMClient 

In [ ]:
from openai import OpenAI

class LLMClient:
    MODELS = {
        'llama3_3': ('nvidia', 'meta/llama-3.3-70b-instruct',          'NVIDIA_API_KEY_LLAMA33'),
        'mamba':    ('nvidia', 'mistralai/mamba-codestral-7b-v0.1',    'NVIDIA_API_KEY_MAMBA'),
        'mistral':  ('nvidia', 'mistralai/mistral-large-3-675b-instruct-2512', 'NVIDIA_API_KEY_MISTRAL'),
        # 'gemma': ('nvidia', 'google/gemma-2-27b-it', 'NVIDIA_API_KEY_GEMMA'),
        'glm': ('nvidia', 'thudm/chatglm3-6b', 'NVIDIA_API_KEY_GLM')
    }
    COSTS = {
        'llama3_3': (0.0, 0.0),
        'mamba':    (0.0, 0.0),
        'mistral':     (0.0, 0.0),
        # 'gemma':    (0.0, 0.0),
        'glm':  (0.0, 0.0)
    }

    def __init__(self, model_key, temperature=0):
        self.model_key   = model_key
        self.provider, self.model_name, api_key_name = self.MODELS[model_key]
        self.temperature = temperature
        self.total_cost  = 0.0
        self.call_log    = []
        self.client = OpenAI(
            base_url='https://integrate.api.nvidia.com/v1',
            api_key=os.environ[api_key_name]
        )

    def call(self, system_prompt, user_prompt, retries=3, retry_delay=60, json_mode=False):
        # For mistral and any model that outputs prose: enforce JSON via system prompt
        if json_mode:
            system_prompt = (
                system_prompt.rstrip() +
                '\n\nCRITICAL: Your response must be valid JSON only. '
                'No prose, no explanation, no markdown fences. '
                'Start your response with { and end with }.'
            )
        for attempt in range(retries):
            try:
                start = time.time()
                resp = self.client.chat.completions.create(
                    model=self.model_name,
                    temperature=self.temperature,
                    max_tokens=3072,
                    stream=False,
                    messages=[
                        {'role': 'system', 'content': system_prompt},
                        {'role': 'user',   'content': user_prompt}
                    ]
                )
                latency = time.time() - start
                tokens_in  = resp.usage.prompt_tokens
                tokens_out = resp.usage.completion_tokens
                metadata = {
                    'tokens_in':    tokens_in,
                    'tokens_out':   tokens_out,
                    'tokens_total': tokens_in + tokens_out,
                    'latency_ms':   round(latency * 1000),
                    'cost_usd':     0.0
                }
                self.call_log.append(metadata)
                return resp.choices[0].message.content, metadata

            except Exception as e:
                err = str(e)
                if '429' in err:
                    print(f'  [RATE LIMIT] Attempt {attempt+1}/{retries} — waiting {retry_delay}s...')
                    time.sleep(retry_delay)
                else:
                    raise e

        raise Exception(f'Rate limit exceeded after {retries} retries')

    def get_stats(self):
        if not self.call_log:
            return {'calls': 0, 'total_tokens': 0, 'total_cost_usd': 0.0, 'avg_latency_ms': 0}
        return {
            'calls':          len(self.call_log),
            'total_tokens':   sum(c['tokens_total'] for c in self.call_log),
            'total_cost_usd': 0.0,
            'avg_latency_ms': sum(c['latency_ms'] for c in self.call_log) / len(self.call_log)
        }

print('LLMClient defined.')

In [ ]:
load_dotenv(override=True)
for model_key in ['llama3_3', 'mamba', 'mistral','glm']:
    try:
        client = LLMClient(model_key, temperature=0)
        resp, meta = client.call("You are helpful.", "Say yes in one word.")
        print(f'{model_key}: {resp.strip()} | {meta["latency_ms"]}ms')
    except Exception as e:
        print(f' {model_key}: {str(e)[:100]}')

## Pipeline Modules (from Stage 2)

In [ ]:
import json
from fuzzywuzzy import fuzz
import re

def _clean_json(text: str) -> str:
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.split('\n')[0].strip() in ['json', 'python', '']:
            text = text.split('\n', 1)[1]
        text = text.rsplit('```', 1)[0]
    return text.strip()

def _parse_with_retry(raw_text, client, system_prompt, user_prompt, retries=2):
    for attempt in range(retries + 1):
        try:
            return json.loads(_clean_json(raw_text))
        except (json.JSONDecodeError, ValueError):
            if attempt < retries:
                raw_text, _ = client.call(system_prompt, user_prompt, json_mode=True)
    raise ValueError('json_parse_failed_after_retries')

def _extract_code(raw: str) -> str:
    """Extract .ino code from LLM output, handling markdown fences."""
    code = raw.strip()
    for pattern in [r'```(?:cpp|c\+\+|arduino|c)?\s*\n(.*?)```',
                    r'```\s*(.*?)```']:
        m = re.search(pattern, code, re.DOTALL | re.IGNORECASE)
        if m:
            code = m.group(1).strip()
            break
    opens = code.count('{')
    closes = code.count('}')
    if opens > closes:
        code += '\n' + ('}' * (opens - closes))
    if 'void setup()' not in code:
        code += '\n\nvoid setup() {\n}\n'
    if 'void loop()' not in code:
        code += '\n\nvoid loop() {\n}\n'
    return code.encode('ascii', errors='ignore').decode()

STOPWORDS = {'the', 'a', 'an', 'and', 'or', 'module', 'sensor', 'shield', 'board'}

def fuzzy_lookup(name, dictionary, use_fuzzy=True):
    if not name:
        return (None, 0)
    entry = dictionary.lookup_by_name(name)
    if entry:
        return (entry, 100)
    if not use_fuzzy:
        return (None, 0)
    name_lower = name.strip().lower()
    cid = dictionary.alias_index.get(name_lower)
    if cid:
        return (dictionary.lookup_by_id(cid), 100)
    best_entry, best_score = None, 0
    for alias_key, cid in dictionary.alias_index.items():
        score = fuzz.token_set_ratio(name_lower, alias_key)
        if score > best_score:
            best_score = score
            best_entry = dictionary.lookup_by_id(cid)
    if best_score >= 75:
        return (best_entry, best_score)
    name_words = set(name_lower.split()) - STOPWORDS
    if not name_words:
        return (None, 0)
    for alias_key, cid in dictionary.alias_index.items():
        alias_words = set(alias_key.split()) - STOPWORDS
        overlap = len(name_words & alias_words)
        if overlap >= 1:
            score = 60 + (overlap * 5)
            if score > best_score:
                best_score = score
                best_entry = dictionary.lookup_by_id(cid)
    return (best_entry, best_score) if best_score >= 65 else (None, 0)

def extract_components(raw_names, dictionary, min_confidence=75, top_k=5,
                        use_fuzzy=True, dedup=True, return_metadata=False):
    if not raw_names:
        return {'component_ids': [], 'component_names': [],
                'component_scores': [], 'raw_extracted_names': []}
    candidates = []
    for name in raw_names:
        entry, score = fuzzy_lookup(name, dictionary, use_fuzzy=use_fuzzy)
        if entry:
            candidates.append({'id': entry['id'], 'name': entry['name'], 'score': score})
    candidates.sort(key=lambda x: x['score'], reverse=True)
    filtered_ids, filtered_names, filtered_scores = [], [], []
    seen = set()
    filtered_out = 0
    for c in candidates:
        is_new = (c['id'] not in seen) if dedup else True
        if c['score'] >= min_confidence and is_new:
            filtered_ids.append(c['id'])
            filtered_names.append(c['name'])
            filtered_scores.append(c['score'])
            seen.add(c['id'])
        elif c['score'] < min_confidence:
            filtered_out += 1
    if top_k > 0 and len(filtered_ids) < top_k:
        for c in candidates:
            if len(filtered_ids) >= top_k:
                break
            if c['id'] not in seen:
                filtered_ids.append(c['id'])
                filtered_names.append(c['name'])
                filtered_scores.append(c['score'])
                seen.add(c['id'])
    result = {
        'component_ids':       filtered_ids,
        'component_names':     filtered_names,
        'component_scores':    filtered_scores,
        'raw_extracted_names': raw_names,
    }
    if return_metadata:
        result['metadata'] = {'filtered_low_confidence': filtered_out,
                               'min_confidence_used': min_confidence}
    return result

# ── LEGACY ALIASES for backward compatibility ──
clean_json = _clean_json
parse_with_retry = _parse_with_retry
import json as _json


In [ ]:
STRICT_EXTRACTION_SYSTEM = """\
You are an expert Arduino systems engineer working with a STRICT evaluation pipeline.

Your job is to:
1) Identify hardware components from a project description
2) Normalize them to a canonical list
3) Extract structured intent in EXACT schema

-------------------------------------
CRITICAL CONSTRAINTS (MUST FOLLOW)
-------------------------------------

1. CANONICAL COMPONENT MATCHING (STRICT BUT FLEXIBLE)

You will be given a canonical component list.

- You MUST map user-described components to the CLOSEST canonical name
- You are allowed to normalize informal names:
    "ultrasonic sensor" \u2192 "HC-SR04 Ultrasonic Sensor"
    "servo motor" \u2192 "Servo Motor"
    "arduino" \u2192 "Arduino Uno"
    "temp sensor" \u2192 "DHT22 Temperature Sensor"

- NEVER output generic names if a specific canonical version exists
- NEVER invent new components not in the list

If unsure, choose the closest valid canonical match.

-------------------------------------

2. OUTPUT FORMAT (STRICT JSON ONLY)

You MUST output EXACTLY this schema:

{
  "components": ["Canonical Component Name"],
  "confidence": {"Canonical Component Name": 0.0},

  "intent": {
    "primary_task": "string",
    "conditions": ["string"],
    "thresholds": {"param": {"value": number, "unit": "string", "operator": "string"}},
    "timing": {"loop_delay_ms": number},
    "inputs": ["string"],
    "outputs": ["string"],
    "communication": ["string"]
  }
}

-------------------------------------

3. FIELD RULES (NO DEVIATION)

- Use EXACT key names (case-sensitive)
- NEVER use:
    \u274c "PRIMARY TASK"
    \u274c "COMPONENTS"
    \u274c "PRIMARY OUTPUT"
- ALWAYS use:
    \u2705 "primary_task"
    \u2705 "components"
    \u2705 "outputs"

-------------------------------------

4. COMPLETENESS REQUIREMENTS

- NEVER leave fields as null
- If unknown, use:
    [] for lists
    {} for objects

Examples:
"thresholds": {}
"timing": {}

-------------------------------------

5. INTENT EXTRACTION RULES

- primary_task \u2192 ONE sentence describing goal
- conditions \u2192 logical triggers (e.g., "distance < 20 cm")
- inputs \u2192 sensors
- outputs \u2192 actuators
- communication \u2192 Serial, I2C, SPI, etc.

-------------------------------------

6. NO EXTRA TEXT

- DO NOT include explanations
- DO NOT include markdown
- DO NOT include comments
- ONLY output valid JSON

-------------------------------------

7. VALIDATION BEFORE OUTPUT (MANDATORY)

Before returning:
- Ensure all component names EXACTLY match canonical list
- Ensure JSON is parseable
- Ensure all required fields exist
- Ensure no null values

-------------------------------------

INPUTS YOU WILL RECEIVE:
- Canonical component list
- Project description

-------------------------------------

OUTPUT:
ONLY VALID JSON
"""

COMPONENT_EXTRACTION_SYSTEM = """\
You are a hardware component identification expert for Arduino and embedded systems.

Given a project description, extract ALL hardware components mentioned or implied.
Return a JSON object: {"components": ["component name", ...]}

Rules:
- Use standard component names (e.g. "DHT22", "Servo Motor", "Ultrasonic Sensor")
- Include every sensor, actuator, display, communication module, and power component
- Include IMPLIED components (a project "with LED indicator" implies LED + resistor)
- Do NOT include: jumper wires, breadboard, USB cable, Arduino board itself
- Output ONLY the JSON object, no explanation

Common aliases:
- I2C LCD / LCD I2C / 16x2 LCD with I2C \u2192 "LCD I2C Display"
- ESP8266 / NodeMCU / WiFi module \u2192 "ESP8266 WiFi Module"
- L298N / motor driver / H-bridge \u2192 "L298N Motor Driver"
- A4988 / DRV8825 / stepper driver \u2192 "Stepper Motor Driver"
- Rotary encoder / KY-040 \u2192 "Rotary Encoder"
- HC-SR04 / sonar / ultrasonic \u2192 "Ultrasonic Sensor"
- OLED / SSD1306 / 128x64 display \u2192 "OLED Display"
"""

INTENT_SYSTEM = """\
You are an embedded systems requirements analyst.

Extract the full intent of the Arduino project from the description.
Return a JSON object \u2014 do NOT skip any key, use null if not applicable:

{
  "PRIMARY_TASK":      string,
  "COMPONENTS":        list,
  "CONDITIONS":        list,
  "TIMING":            string,
  "THRESHOLDS":        object,
  "PRIMARY_OUTPUT":    string,
  "SECONDARY_OUTPUT":  string,
  "completeness_score": float
}

Output ONLY the JSON object, no explanation.
"""

MAPPING_SYSTEM = """\
You are an Arduino hardware mapping expert.

You are given:
1. Project intent (JSON)
2. Hardware specifications for each component (structured text with pin wiring info)

Your job: assign specific Arduino pin numbers and verify feasibility.

Return a JSON object:
{
  "feasible":        bool,
  "reason":          string,
  "pin_assignments": {
    "COMPONENT_SIGNAL": pin_number,
    ...
  },
  "conflicts": []
}

Arduino Uno pin constraints \u2014 MUST follow these:
- Digital pins available: 2\u201313 (avoid 0,1 \u2014 used by Serial)
- Analog input pins: A0\u2013A5 (use integers 14\u201319 in pin_assignments, or "A0"\u2013"A5")
- PWM-capable pins ONLY: 3, 5, 6, 9, 10, 11
- I2C: SDA=A4, SCL=A5 (shared \u2014 only one I2C bus)
- SPI: MOSI=11, MISO=12, SCK=13, SS=10
- Servo signals: use PWM pins (9 or 10 preferred)
"""

PLANNING_SYSTEM = """\
You are an Arduino code architect.

You are given:
1. Hardware mapping (pin assignments JSON)
2. Hardware specifications (which list EXACT required library names and Arduino functions)

CRITICAL: Use ONLY the library names listed in the hardware specifications.
Do NOT invent library names. If the spec says "LiquidCrystal_I2C", use that exactly.

Return a JSON object:
{
  "libraries":    [list of library names WITHOUT .h \u2014 taken verbatim from specs],
  "pin_defines":  ["#define TRIG_PIN 9", "#define ECHO_PIN 10", ...],
  "global_vars":  ["Servo myServo;", "DHT dht(DHT_PIN, DHT22);", ...],
  "setup_steps":  ["Serial.begin(9600);", "myServo.attach(SERVO_PIN);", ...],
  "loop_steps":   ["Read sensor value", "Check threshold", "Actuate output", ...],
  "helper_funcs": ["float readDistance() {...}", ...]
}
"""

SYNTHESIS_SYSTEM = """\
You are an expert Arduino programmer generating production-quality .ino code.

Rules:
- Output ONLY raw .ino code \u2014 no markdown fences, no prose explanation
- Always include setup() and loop() functions
- Add brief inline comments on non-obvious lines
- Use EXACTLY the pin numbers and library names from the code plan
- Do NOT add libraries not in the plan
- Do NOT change pin numbers from the plan
"""

def build_synthesis_prompt(plan: dict, rag_context: str, component_ids: list,
                            dictionary: HardwareDictionary) -> str:
    rag_libs = []
    for cid in component_ids:
        entry = dictionary.lookup_by_id(cid)
        if entry: rag_libs.extend(entry['libraries'])
    seen = set()
    rag_libs_deduped = []
    for lib in rag_libs:
        if lib and lib.lower() not in seen:
            seen.add(lib.lower())
            rag_libs_deduped.append(lib)
    plan_libs = plan.get('libraries', [])
    for lib in plan_libs:
        if lib and lib.lower() not in seen:
            seen.add(lib.lower())
            rag_libs_deduped.append(lib)
    if rag_libs_deduped:
        include_block = '\n'.join(f'#include <{lib}.h>' if not lib.endswith('.h') else f'#include <{lib}>' for lib in rag_libs_deduped)
    else:
        include_block = '// No external libraries required'
    pin_defines = plan.get('pin_defines', [])
    if pin_defines:
        define_block = '\n'.join(pin_defines)
    else:
        pin_assignments = plan.get('pin_assignments', {})
        define_block = '\n'.join(f'#define {sig.upper().replace(" ", "_")}_PIN {pin}' for sig, pin in pin_assignments.items())
    return f"""Hardware specifications:\n{rag_context}\n\nCode plan:\n{json.dumps(plan, indent=2)}\n\nGenerate the complete Arduino sketch.\nYour code MUST start with exactly these lines:\n```\n{include_block}\n\n{define_block}\n```\n\nContinue with the full sketch body. Output raw .ino code only."""

def understand_intent(description, client):
    raw, meta = client.call(INTENT_SYSTEM, f'Project description:\n{description}', json_mode=True)
    try:
        data = json.loads(_clean_json(raw))
    except (json.JSONDecodeError, ValueError):
        data = {}
    return {
        'intent':             raw,
        'intent_parsed':      data,
        'completeness_score': float(data.get('completeness_score', 0.0)),
        'metadata':           meta,
    }

def hardware_mapping(intent_parsed: dict, rag_context: str, client):
    user_prompt = f'Project intent:\n{json.dumps(intent_parsed, indent=2)}\n\nHardware specifications:\n{rag_context}'
    raw, meta = client.call(MAPPING_SYSTEM, user_prompt, json_mode=True)
    try:
        mapping = _parse_with_retry(raw, client, MAPPING_SYSTEM, user_prompt, retries=2)
    except ValueError:
        return {'error': 'json_parse_failed_after_retries', 'raw': raw, 'metadata': meta}
    return {'mapping': mapping, 'metadata': meta}

def code_planning(mapping: dict, rag_context: str, client):
    user_prompt = f'Hardware mapping:\n{json.dumps(mapping, indent=2)}\n\nHardware specifications:\n{rag_context}'
    raw, meta = client.call(PLANNING_SYSTEM, user_prompt, json_mode=True)
    try:
        plan = json.loads(_clean_json(raw))
    except (json.JSONDecodeError, ValueError):
        plan = {}
    return {'plan': plan, 'raw': raw, 'metadata': meta}

def synthesize_code(plan: dict, rag_context: str, client,
                    component_ids: list, dictionary: HardwareDictionary):
    user_prompt = build_synthesis_prompt(plan, rag_context, component_ids, dictionary)
    raw, meta = client.call(SYNTHESIS_SYSTEM, user_prompt, json_mode=False)
    code = _extract_code(raw)
    return {'code': code, 'metadata': meta}

print('System prompts and module functions defined.')

## Full Pipeline + Batch Runner

In [ ]:
STRICT_EXTRACTION_SYSTEM = """\
You are an expert Arduino systems engineer working with a STRICT evaluation pipeline.

Your job is to:
1) Identify hardware components from a project description
2) Normalize them to a canonical list
3) Extract structured intent in EXACT schema

-------------------------------------
CRITICAL CONSTRAINTS (MUST FOLLOW)
-------------------------------------

1. CANONICAL COMPONENT MATCHING (STRICT BUT FLEXIBLE)

You will be given a canonical component list.

- You MUST map user-described components to the CLOSEST canonical name
- You are allowed to normalize informal names:
    "ultrasonic sensor" \u2192 "HC-SR04 Ultrasonic Sensor"
    "servo motor" \u2192 "Servo Motor"
    "arduino" \u2192 "Arduino Uno"
    "temp sensor" \u2192 "DHT22 Temperature Sensor"

- NEVER output generic names if a specific canonical version exists
- NEVER invent new components not in the list

If unsure, choose the closest valid canonical match.

-------------------------------------

2. OUTPUT FORMAT (STRICT JSON ONLY)

You MUST output EXACTLY this schema:

{
  "components": ["Canonical Component Name"],
  "confidence": {"Canonical Component Name": 0.0},

  "intent": {
    "primary_task": "string",
    "conditions": ["string"],
    "thresholds": {"param": {"value": number, "unit": "string", "operator": "string"}},
    "timing": {"loop_delay_ms": number},
    "inputs": ["string"],
    "outputs": ["string"],
    "communication": ["string"]
  }
}

-------------------------------------

3. FIELD RULES (NO DEVIATION)

- Use EXACT key names (case-sensitive)
- NEVER use:
    \u274c "PRIMARY TASK"
    \u274c "COMPONENTS"
    \u274c "PRIMARY OUTPUT"
- ALWAYS use:
    \u2705 "primary_task"
    \u2705 "components"
    \u2705 "outputs"

-------------------------------------

4. COMPLETENESS REQUIREMENTS

- NEVER leave fields as null
- If unknown, use:
    [] for lists
    {} for objects

Examples:
"thresholds": {}
"timing": {}

-------------------------------------

5. INTENT EXTRACTION RULES

- primary_task \u2192 ONE sentence describing goal
- conditions \u2192 logical triggers (e.g., "distance < 20 cm")
- inputs \u2192 sensors
- outputs \u2192 actuators
- communication \u2192 Serial, I2C, SPI, etc.

-------------------------------------

6. NO EXTRA TEXT

- DO NOT include explanations
- DO NOT include markdown
- DO NOT include comments
- ONLY output valid JSON

-------------------------------------

7. VALIDATION BEFORE OUTPUT (MANDATORY)

Before returning:
- Ensure all component names EXACTLY match canonical list
- Ensure JSON is parseable
- Ensure all required fields exist
- Ensure no null values

-------------------------------------

INPUTS YOU WILL RECEIVE:
- Canonical component list
- Project description

-------------------------------------

OUTPUT:
ONLY VALID JSON
"""

MAX_RAG_CHARS = {
    'mamba':    5000,
    'glm':      5000,
    'llama3_3': 14000,
    'mistral':  14000,
}


def run_full_pipeline(description, dictionary, client, verbose=True):
    """
    Runs the full hardware-grounded code synthesis pipeline.
    Combined M1 (Extraction) & M3A (Intent) following STRICT specification.
    """
    import json
    # 1. Combined Extraction & Intent
    canonical_list_str = "\n".join([f"- {name}" for name in dictionary.get_canonical_names_list()])
    
    extraction_input = f"""Canonical component list:
{canonical_list_str}

Project description:
{description}"""

    raw_combined, meta = client.call(STRICT_EXTRACTION_SYSTEM, extraction_input, json_mode=True)
    
    if verbose:
        print("RAW LLM OUTPUT (M1+M3A):\n", raw_combined)
        
    try:
        combined_data = json.loads(_clean_json(raw_combined))
        raw_names = combined_data.get('components', [])
        intent_parsed = combined_data.get('intent', {})
    except (json.JSONDecodeError, ValueError) as e:
        if verbose: print(f"PARSE ERROR: {e}\nRaw: {raw_combined}")
        raw_names = []
        intent_parsed = {}

    # 2. Extract components (Safety Net Fuzzy Matching)
    extract_result = extract_components(raw_names, dictionary, min_confidence=75, top_k=5)
    component_ids = extract_result.get('component_ids', [])
    
    if verbose:
        if not component_ids:
            print('  No components extracted \u2014 continuing with empty context...')
        else:
            print(f"  Extracted: {extract_result['component_names']}")

    # 3. Hardware Mapping (M2)
    mapping_result = hardware_mapping(intent_parsed, component_ids, dictionary, client)
    pin_table = mapping_result.get('pin_table_comment', '// No pins defined')

    # 4. Code Planning (M3B)
    plan_result = code_planning(intent_parsed, component_ids, dictionary, client)
    planning_text = plan_result.get('plan', '// No plan generated')

    # 5. Code Synthesis (M4)
    # Build RAG Context
    rag_context = dictionary.build_rag_context(component_ids)
    
    final_result = synthesize_code(intent_parsed, planning_text, pin_table, rag_context, client)
    
    return {
        'code': final_result.get('code', ''),
        'metadata': {
            'extraction_meta': extract_result.get('metadata', {}),
            'mapping_meta': mapping_result.get('metadata', {}),
            'plan_meta': plan_result.get('metadata', {}),
            'synthesis_meta': final_result.get('metadata', {}),
            'raw_combined': raw_combined
        },
        'component_ids': component_ids,
        'intent_parsed': intent_parsed
    }

print('run_full_pipeline defined (RAG Rewrite).')

In [ ]:
import re

def clean_arduino_code(raw_output: str) -> str:
    """
    Cleans LLM output into valid Arduino (.ino) code.
    Handles markdown, junk text, missing structure, and truncation.
    """
    if not raw_output or not isinstance(raw_output, str):
        return ""

    code = raw_output.strip()

    # ── 1. Extract code from markdown blocks ───────────────────────
    code_block_patterns = [
        r"```cpp\s*(.*?)```",
        r"```c\+\+\s*(.*?)```",
        r"```\s*(.*?)```"
    ]

    for pattern in code_block_patterns:
        match = re.search(pattern, code, re.DOTALL | re.IGNORECASE)
        if match:
            code = match.group(1).strip()
            break

    # ── 2. Remove leading junk before actual code ──────────────────
    lines = code.splitlines()
    cleaned_lines = []

    start_keywords = ("#include", "void", "int", "float", "bool", "char")

    started = False
    for line in lines:
        stripped = line.strip()

        if not started:
            if stripped.startswith(start_keywords):
                started = True
                cleaned_lines.append(line)
        else:
            cleaned_lines.append(line)

    if cleaned_lines:
        code = "\n".join(cleaned_lines)

    # ── 3. Remove trailing markdown artifacts ──────────────────────
    code = re.sub(r"```.*$", "", code, flags=re.DOTALL).strip()

    # ── 4. Ensure setup() and loop() exist ─────────────────────────
    if "void setup()" not in code:
        code += "\n\nvoid setup() {\n}\n"

    if "void loop()" not in code:
        code += "\n\nvoid loop() {\n}\n"

    # ── 5. Fix common truncation issues ────────────────────────────
    # Balance braces
    open_braces = code.count("{")
    close_braces = code.count("}")

    if open_braces > close_braces:
        code += "\n" + ("}" * (open_braces - close_braces))

    # ── 6. Remove non-ASCII garbage (optional but helpful) ─────────
    code = code.encode("ascii", errors="ignore").decode()

    return code

def run_batch_pipeline(projects, model_key='llama3_3', temperature=0, save_outputs=True, sleep_sec=3):
    results = []
    out_dir = f'{OUTPUT_DIR}/{model_key}'
    os.makedirs(out_dir, exist_ok=True)

    for proj in tqdm(projects, desc=f'Running {model_key}'):
        pid = proj['project_id']

        # ── CHECKPOINT: skip if already completed ──────────────────
        result_path = f'{out_dir}/project_{pid}_result.json'
        if os.path.exists(result_path):
            with open(result_path) as f:
                results.append(json.load(f))
            continue

        desc = proj.get('description', '').strip()
        if not desc:
            code_path = proj.get('code_path', '')
            if code_path and os.path.exists(code_path):
                with open(code_path, 'r', encoding='utf-8', errors='ignore') as f:
                    desc = f.read().strip()
            else:
                print(f'  [SKIP] Project {pid}: no description or code_path')
                results.append({'project_id': pid, 'error': 'no description'})
                continue

        MAX_INPUT_CHARS = 6000
        if len(desc) > MAX_INPUT_CHARS:
            desc = desc[:MAX_INPUT_CHARS]
            print(f'  [TRUNCATED] Project {pid}: input truncated to {MAX_INPUT_CHARS} chars')

        print(f'\n── Project {pid} ──')
        try:
            pipeline_result = run_full_pipeline(desc, model_key, temperature)
            pipeline_result['project_id'] = pid
            pipeline_result['complexity'] = proj.get('complexity', 'Unknown')
            results.append(pipeline_result)

            if save_outputs:
                # Always save .ino — use stub if no code generated
                code = (pipeline_result.get('module3d') or {}).get('code', '')
                if code:
                    ino_content = clean_arduino_code(code)
                else:
                    ino_content = f'// Project {pid}: code generation failed\nvoid setup(){{}}\nvoid loop(){{}}'
                with open(f'{out_dir}/project_{pid}.ino', 'w') as f:
                    f.write(ino_content)
                # Always save result JSON
                with open(result_path, 'w') as f:
                    json.dump(pipeline_result, f, indent=2)

        except Exception as e:
            err_msg = str(e)
            print(f'  Error on project {pid}: {err_msg}')
            if '429' in err_msg:
                print(f'\n  [RATE LIMIT] Stopping {model_key}. Re-run to resume from project {pid}.')
                break
            results.append({'project_id': pid, 'error': err_msg})

        time.sleep(sleep_sec)

    summary = {
        'model': model_key,
        'temperature': temperature,
        'total_projects': len(projects),
        'completed': sum(1 for r in results if 'error' not in r),
        'failed': sum(1 for r in results if 'error' in r),
        'results': results
    }
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(f'{OUTPUT_DIR}/{model_key}_batch_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    return summary

print('run_batch_pipeline defined.')

## Run Batch: All 3 Models
Runs on all in-scope projects. Adjust `[:5]` to run a subset for testing.

In [ ]:
# MODELS_TO_RUN = ['llama3_3', 'mamba', 'mistral']
# # all_summaries = {}

In [ ]:
MODELS_TO_RUN = ['llama3_3', 'mamba', 'mistral','glm']
all_summaries = {}

for model_key in MODELS_TO_RUN:
    print(f'\n══════════════════════════════════════')
    print(f'  Running model: {model_key}')
    print(f'══════════════════════════════════════')
    summary = run_batch_pipeline(
        in_scope,
        model_key=model_key,
        temperature=0,
        save_outputs=True,
        sleep_sec=2
    )
    all_summaries[model_key] = summary
    print(f'\n  {model_key} DONE: {summary["completed"]}/{summary["total_projects"]} completed')

print('\nAll models done.')

In [ ]:
# Check if any files saved yet
for model_key in ['llama3_3', 'mamba', 'mistral']:
    out_dir = f'{OUTPUT_DIR}/{model_key}'
    if os.path.exists(out_dir):
        completed = len([f for f in os.listdir(out_dir) if f.endswith('_result.json')])
        print(f'{model_key}: {completed}/434 saved')
    else:
        print(f'{model_key}: no output dir yet')

## Scoring: Physical Metrics

In [ ]:
def compute_basic_metrics(pred_ids, gt_ids):
    """
    Input:
        pred_ids: list[int]
        gt_ids: list[int]

    Output:
        dict with:
            tp, fp, fn,
            precision, recall, f1,
            exact_match
    """

    # 1. Deduplicate
    pred = set(pred_ids or [])
    gt   = set(gt_ids or [])

    # 2. Compute sets
    tp = len(pred & gt)
    fp = len(pred - gt)
    fn = len(gt - pred)

    # 3. Edge case: both empty
    if len(pred) == 0 and len(gt) == 0:
        return {
            "tp": 0, "fp": 0, "fn": 0,
            "precision": 1.0,
            "recall": 1.0,
            "f1": 1.0,
            "exact_match": 1
        }

    # 4. Metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    exact_match = 1 if pred == gt else 0

    # 5. Validation
    assert 0 <= precision <= 1
    assert 0 <= recall <= 1
    assert 0 <= f1 <= 1

    return {
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match": exact_match
    }

def normalize_name(name):
    remove_words = ["sensor", "module", "device"]
    tokens = name.lower().split()
    return [t for t in tokens if t not in remove_words]


def is_similar(name1, name2):
    t1 = set(normalize_name(name1))
    t2 = set(normalize_name(name2))

    if not t1 or not t2:
        return False

    return (
        t1 & t2 or
        " ".join(t1) in " ".join(t2) or
        " ".join(t2) in " ".join(t1)
    )


def compute_relaxed_metrics(pred_ids, gt_ids, id_to_name):
    pred = list(set(pred_ids or []))
    gt   = list(set(gt_ids or []))

    matched_gt = set()
    tp = 0

    # 1. Exact matches first
    for p in pred:
        if p in gt and p not in matched_gt:
            tp += 1
            matched_gt.add(p)

    # 2. Similar matches
    for p in pred:
        if p in matched_gt:
            continue

        for g in gt:
            if g in matched_gt:
                continue

            if is_similar(id_to_name[p], id_to_name[g]):
                tp += 1
                matched_gt.add(g)
                break

    fp = len(pred) - tp
    fn = len(gt) - tp

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return precision, recall, f1

def score_all_projects(results, id_to_name):

    rows = []

    total_tp = total_fp = total_fn = 0
    total_relaxed_p = total_relaxed_r = total_relaxed_f1 = 0

    num_fn = num_fp = semantic_errors = 0

    exp_tp = exp_fp = exp_fn = 0
    inf_tp = inf_fp = inf_fn = 0
    has_stage_split = False

    for proj in results:

        pred = list(set(proj.get("predicted", [])))
        gt   = list(set(proj.get("ground_truth", [])))

        metrics = compute_basic_metrics(pred, gt)

        total_tp += metrics["tp"]
        total_fp += metrics["fp"]
        total_fn += metrics["fn"]

        # Relaxed
        rp, rr, rf1 = compute_relaxed_metrics(pred, gt, id_to_name)

        total_relaxed_p += rp
        total_relaxed_r += rr
        total_relaxed_f1 += rf1

        # Failure analysis
        fn_set = set(gt) - set(pred)
        fp_set = set(pred) - set(gt)

        num_fn += len(fn_set)
        num_fp += len(fp_set)

        for fp_id in fp_set:
            for gt_id in gt:
                if is_similar(id_to_name[fp_id], id_to_name[gt_id]):
                    semantic_errors += 1
                    break

        # Stage-wise
        explicit = proj.get("explicit_components")
        inferred = proj.get("inferred_components")

        if explicit is not None and inferred is not None:
            has_stage_split = True

            exp_m = compute_basic_metrics(explicit, gt)
            inf_m = compute_basic_metrics(inferred, gt)

            exp_tp += exp_m["tp"]
            exp_fp += exp_m["fp"]
            exp_fn += exp_m["fn"]

            inf_tp += inf_m["tp"]
            inf_fp += inf_m["fp"]
            inf_fn += inf_m["fn"]

        rows.append({
            **metrics,
            "relaxed_f1": rf1
        })

    # Macro
    n = len(rows) if rows else 1
    avg_precision = sum(r["precision"] for r in rows) / n
    avg_recall    = sum(r["recall"] for r in rows) / n
    avg_f1        = sum(r["f1"] for r in rows) / n
    avg_exact     = sum(r["exact_match"] for r in rows) / n

    # Micro
    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0

    print("\n=== FAILURE ANALYSIS ===")
    print("Avg Missing (FN):", num_fn / n)
    print("Avg Extra (FP):", num_fp / n)
    print("Semantic Errors:", semantic_errors)

    if has_stage_split:
        print("\n=== STAGE-WISE ===")
        print("Explicit Recall:", exp_tp / (exp_tp + exp_fn) if (exp_tp + exp_fn) else 0)
        print("Inferred Recall:", inf_tp / (inf_tp + inf_fn) if (inf_tp + inf_fn) else 0)

    return {
        "avg_precision": avg_precision,
        "avg_recall": avg_recall,
        "avg_f1": avg_f1,
        "avg_exact_match": avg_exact,
        "micro_f1": micro_f1
    }


##  Arduino CLI Compilation Check

In [ ]:
import tempfile, shutil

def check_compilation(ino_path, fqbn='arduino:avr:uno', timeout=60):
    """
    Run Arduino CLI compilation. Wraps .ino in a correctly named sketch folder.
    """
    sketch_name = os.path.splitext(os.path.basename(ino_path))[0]
    
    with tempfile.TemporaryDirectory() as tmpdir:
        # Arduino CLI requires: sketch_name/sketch_name.ino
        sketch_dir = os.path.join(tmpdir, sketch_name)
        os.makedirs(sketch_dir)
        shutil.copy(ino_path, os.path.join(sketch_dir, f'{sketch_name}.ino'))
        
        result = subprocess.run(
            ['arduino-cli', 'compile', '--fqbn', fqbn, sketch_dir],
            capture_output=True, text=True, timeout=timeout
        )
    
    return {
        'success': result.returncode == 0,
        'stdout': result.stdout,
        'stderr': result.stderr
    }


def run_compilation_checks(model_key):
    """Run Arduino CLI on all generated .ino files for a model"""
    out_dir = f'{OUTPUT_DIR}/{model_key}'
    if not os.path.exists(out_dir):
        print(f'No output dir for {model_key}')
        return {}

    compilation_results = {}
    ino_files = [f for f in os.listdir(out_dir) if f.endswith('.ino')]
    print(f'Found {len(ino_files)} .ino files for {model_key}')

    for fname in tqdm(ino_files, desc=f'Compiling {model_key}'):
        pid = fname.replace('project_', '').replace('.ino', '')
        path = os.path.join(out_dir, fname)
        result = check_compilation(path)
        success = result['success']
        error = result['stderr']
        compilation_results[pid] = {'success': success, 'error': error}
        if not success and error:
            error_lines = [l for l in error.splitlines() if 'error:' in l.lower() or 'warning:' in l.lower()]
            error_summary = error_lines[0][:100] if error_lines else error.splitlines()[-1][:100]
        else:
            error_summary = ''
        print(f'  Project {pid}: {"✅" if success else "❌"} {error_summary}')

    # Save results
    with open(f'{OUTPUT_DIR}/{model_key}_compilation.json', 'w') as f:
        json.dump(compilation_results, f, indent=2)

    passed = sum(1 for v in compilation_results.values() if v['success'])
    print(f'\n{model_key}: {passed}/{len(compilation_results)} compiled successfully')
    return compilation_results


print('Compilation check functions defined.')

## Run Compilation Checks (All Models)

In [ ]:
# ── Pre-Compilation Cleaner ───────────────────────────────────────────────────
# Applies clean_arduino_code() to every saved .ino file before arduino-cli runs.
# Fixes: markdown fences, leading prose, missing setup()/loop(), unbalanced braces,
# non-ASCII chars, and trailing markdown artifacts.

import os, re

def clean_arduino_code_v2(raw_output: str) -> str:
    if not raw_output or not isinstance(raw_output, str):
        return ""

    code = raw_output.strip()

    # 1. Extract code from markdown fences (cpp / c++ / arduino / plain)
    fence_match = re.search(
        r"```(?:cpp|c\+\+|arduino|c)?\s*\n(.*?)```",
        code, re.DOTALL | re.IGNORECASE
    )
    if fence_match:
        code = fence_match.group(1).strip()
    else:
        code = re.sub(r"^```[\w]*\n", "", code, flags=re.MULTILINE).strip()

    # 2. Remove any remaining trailing ``` or ~~~
    code = re.sub(r"[`~]{3}\s*$", "", code, flags=re.MULTILINE).strip()

    # 3. Drop leading non-code lines (prose before first real directive)
    code_start_re = re.compile(
        r"^(#include|#define|#pragma|void\s|int\s|float\s|bool\s|char\s|const\s|unsigned\s|long\s|byte\s|String\s|//)",
        re.IGNORECASE
    )
    lines = code.splitlines()
    start_idx = 0
    for idx, line in enumerate(lines):
        if code_start_re.match(line.strip()):
            start_idx = idx
            break
    code = "\n".join(lines[start_idx:])

    # 4. Strip non-ASCII characters
    code = code.encode('ascii', errors='ignore').decode()

    # 5. Ensure setup() and loop() exist
    if 'void setup()' not in code and 'void setup ()' not in code:
        code += '\n\nvoid setup() {\n  // auto-added by cleaner\n}\n'
    if 'void loop()' not in code and 'void loop ()' not in code:
        code += '\n\nvoid loop() {\n  // auto-added by cleaner\n}\n'

    # 6. Balance unmatched opening braces from LLM truncation
    open_b  = code.count('{')
    close_b = code.count('}')
    if open_b > close_b:
        code += '\n' + '}' * (open_b - close_b)

    return code.strip()


def clean_all_ino_files(output_dir, models, dry_run=False):
    summary = {}
    for model_key in models:
        model_dir = os.path.join(output_dir, model_key)
        if not os.path.isdir(model_dir):
            print(f'[SKIP] {model_key}: directory not found ({model_dir})')
            summary[model_key] = {'total': 0, 'changed': 0, 'skipped': 0}
            continue

        ino_files = sorted(f for f in os.listdir(model_dir) if f.endswith('.ino'))
        total, changed, skipped = len(ino_files), 0, 0
        print(f'\n[{model_key}] Cleaning {total} .ino files...')

        for fname in ino_files:
            path = os.path.join(model_dir, fname)
            try:
                with open(path, 'r', encoding='utf-8', errors='ignore') as fh:
                    raw = fh.read()
                cleaned = clean_arduino_code_v2(raw)
                if cleaned == raw.strip():
                    skipped += 1
                    continue
                if not dry_run:
                    with open(path, 'w', encoding='utf-8') as fh:
                        fh.write(cleaned)
                changed += 1
            except Exception as e:
                print(f'  [ERROR] {fname}: {e}')
                skipped += 1

        print(f'  Done — changed: {changed}, already clean: {skipped}, total: {total}')
        summary[model_key] = {'total': total, 'changed': changed, 'skipped': skipped}
    return summary


# ── Run ───────────────────────────────────────────────────────────────────────
cleaning_summary = clean_all_ino_files(
    output_dir=OUTPUT_DIR,
    models=MODELS_TO_RUN,
    dry_run=False   # set True to preview without overwriting
)

print('\n── Cleaning Summary ────────────────────────────────')
for model, stats in cleaning_summary.items():
    pct = round(stats['changed'] / stats['total'] * 100, 1) if stats['total'] else 0
    print(f"{model:15s}  total={stats['total']:4d}  cleaned={stats['changed']:4d} ({pct}%)  already_ok={stats['skipped']:4d}")
print('Pre-compilation cleaning complete. Ready to compile.')

In [ ]:
compilation_all = {}
for model_key in MODELS_TO_RUN:
    compilation_all[model_key] = run_compilation_checks(model_key)

## Score All Models & Merge Compilation Results

In [ ]:
scored_dfs = {}
for model_key in MODELS_TO_RUN:
    summary = all_summaries[model_key]
    df = score_all_projects(summary, gt_lookup)
    df['model'] = model_key
    scored_dfs[model_key] = df
    print(f'{model_key}: {len(df)} projects scored.')

all_scores = pd.concat(scored_dfs.values(), ignore_index=True)

# Add Normalized Error Rates
all_scores['over_spec_rate'] = all_scores['fp'] / (all_scores['fp'] + all_scores['tp'] + 1e-6)
all_scores['under_spec_rate'] = all_scores['fn'] / (all_scores['fn'] + all_scores['tp'] + 1e-6)

all_scores.to_csv(f'{OUTPUT_DIR}/all_scores_final.csv', index=False)
print(f'\nSaved all_scores_final.csv ({len(all_scores)} rows)')

## Model Comparison Summary

In [ ]:
summary_cols = ['f1', 'lib_f1', 'pin_validity', 'intent_completeness']

if 'compilation_success' in all_scores.columns:
    summary_cols.append('compilation_success')

comparison = all_scores.groupby('model')[summary_cols].mean().round(3)

if 'compilation_success' in comparison.columns:
    comparison['compilation_success'] = comparison['compilation_success'].apply(
        lambda x: f'{x*100:.1f}%' if pd.notnull(x) else 'N/A'
    )

print('\n=== Model Comparison (Mean Scores) ===')
display(comparison)

comparison.to_csv(f'{OUTPUT_DIR}/model_comparison.csv')
print('Saved model_comparison.csv')

## Intra-Family Comparison: Llama 3.1 vs 3.3

In [ ]:
llama_df = all_scores[all_scores['model'].isin(['llama3_1', 'llama3_3'])].copy()

print('=== Llama 3.1 vs 3.3 — Per-Metric Delta ===')
llama_pivot = llama_df.groupby('model')[summary_cols[:-1]].mean().round(3)
display(llama_pivot)

if 'llama3_3' in llama_pivot.index and 'llama3_1' in llama_pivot.index:
    delta = llama_pivot.loc['llama3_3'] - llama_pivot.loc['llama3_1']
    print('\nDelta (3.3 - 3.1):')
    print(delta.round(3))

# By complexity
print('\n=== Llama 3.1 vs 3.3 — By Complexity ===')
display(llama_df.groupby(['model', 'complexity'])['f1'].mean().round(3).unstack())

## Ablation Study (Task 7 Setup)
Compares 3 setups on a subset of projects:
- **Setup A:** Single-agent (no RAG)
- **Setup B:** Single-agent + RAG
- **Setup C:** Multi-agent + RAG (full pipeline)

In [ ]:
SINGLE_AGENT_SYSTEM = """You are an Arduino expert.
Given a project description, generate complete compilable Arduino code.
Output ONLY raw .ino code — no markdown, no explanation.
Include setup() and loop() functions."""

def run_single_agent(description, model_key='llama3_3', temperature=0):
    """Setup A: Single-agent, no RAG"""
    client = LLMClient(model_key, temperature)
    raw, meta = client.call(SINGLE_AGENT_SYSTEM, f'Project description:\n{description}')
    code = raw.strip()
    if '```' in code:
        code = code.split('```')[1]
        if code.split('\n')[0].strip() in ['cpp', 'arduino', 'c', '']:
            code = code.split('\n', 1)[1]
        code = code.rsplit('```', 1)[0]
    return {'code': code.strip(), 'metadata': meta, 'setup': 'single_agent'}


def run_single_agent_with_rag(description, dictionary, model_key='llama3_3', temperature=0):
    """Setup B: Single-agent + RAG"""
    client = LLMClient(model_key, temperature)
    # Extract components first
    raw_comp, _ = client.call(COMPONENT_EXTRACTION_SYSTEM, description, json_mode=True)
    try:
        comp_data = _json.loads(clean_json(raw_comp))
        raw_names = comp_data.get('components', [])
    except (_json.JSONDecodeError, ValueError):
        raw_names = []
    extract_result = extract_components(raw_names, dictionary, min_confidence=85)
    component_ids  = extract_result.get('component_ids', [])
    rag_context    = dictionary.build_rag_context(component_ids) if component_ids else ''

    user_prompt = f"""Project description:\n{description}

Hardware specifications:\n{rag_context}

Generate complete Arduino code."""
    raw, meta = client.call(SINGLE_AGENT_SYSTEM, user_prompt)
    code = raw.strip()
    if '```' in code:
        code = code.split('```')[1]
        if code.split('\n')[0].strip() in ['cpp', 'arduino', 'c', '']:
            code = code.split('\n', 1)[1]
        code = code.rsplit('```', 1)[0]
    return {'code': code.strip(), 'metadata': meta,
            'component_ids': component_ids, 'setup': 'single_agent_rag'}


print('Ablation setup functions defined.')

## Run Ablation on Subset

In [ ]:
ABLATION_MODEL  = 'llama3_3'   # run ablation on best-performing model
ABLATION_SUBSET = in_scope[:10]  # adjust size as needed

ablation_results = []

for proj in tqdm(ABLATION_SUBSET, desc='Ablation'):
    pid  = proj['project_id']
    desc = proj.get('description', '').strip()
    if not desc:
        code_path = proj.get('code_path', '')
        if code_path and os.path.exists(code_path):
            with open(code_path, 'r', encoding='utf-8', errors='ignore') as f:
                desc = f.read().strip()

    if not desc:
        continue

    gt = gt_lookup.get(pid, {})
    gt_ids  = gt.get('ground_truth_component_ids', [])
    gt_libs = gt.get('ground_truth_libraries', [])

    row = {'project_id': pid, 'complexity': proj.get('complexity', 'Unknown')}

    # Setup A: Single-agent
    sa = run_single_agent(desc, ABLATION_MODEL)
    lib_a = score_library_correctness(sa['code'], gt_libs)
    row['sa_lib_f1'] = lib_a['f1']
    row['sa_has_code'] = bool(sa['code'])

    # Save .ino
    abl_dir = f'{OUTPUT_DIR}/ablation/{ABLATION_MODEL}'
    os.makedirs(abl_dir, exist_ok=True)
    with open(f'{abl_dir}/project_{pid}_sa.ino', 'w') as f:
        f.write(sa['code'])

    time.sleep(3)

    # Setup B: Single-agent + RAG
    sb = run_single_agent_with_rag(desc, dictionary, ABLATION_MODEL)
    lib_b = score_library_correctness(sb['code'], gt_libs)
    # Filter phantom IDs from predictions only
    comp_b = score_component_correctness(
        [cid for cid in sb.get('component_ids', []) if cid not in PHANTOM_IDS], gt_ids)
    row['sb_lib_f1']  = lib_b['f1']
    row['sb_comp_f1'] = comp_b['f1']
    row['sb_has_code'] = bool(sb['code'])
    with open(f'{abl_dir}/project_{pid}_sb.ino', 'w') as f:
        f.write(sb['code'])

    time.sleep(3)

    # Setup C: Multi-agent + RAG (use full pipeline result from Cell 8)
    full_result = next(
        (r for r in all_summaries[ABLATION_MODEL]['results'] if r.get('project_id') == pid), None
    )
    if full_result:
        code_c     = (full_result.get('module3d') or {}).get('code', '')
        pred_ids_c = (full_result.get('module1') or {}).get('component_ids', [])
        lib_c   = score_library_correctness(code_c, gt_libs)
        # Filter phantom IDs from predictions only
        comp_c  = score_component_correctness(
            [cid for cid in pred_ids_c if cid not in PHANTOM_IDS], gt_ids)
        row['sc_lib_f1']  = lib_c['f1']
        row['sc_comp_f1'] = comp_c['f1']
        row['sc_has_code'] = bool(code_c)

    ablation_results.append(row)

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(f'{OUTPUT_DIR}/ablation_results.csv', index=False)
print('\nAblation complete.')
display(ablation_df)

## Ablation Summary

In [ ]:
abl_summary = {
    'Setup A (single-agent)':       ablation_df[['sa_lib_f1']].mean().rename({'sa_lib_f1': 'lib_f1'}),
    'Setup B (single-agent + RAG)': ablation_df[['sb_lib_f1', 'sb_comp_f1']].mean().rename({'sb_lib_f1': 'lib_f1', 'sb_comp_f1': 'comp_f1'}),
    'Setup C (multi-agent + RAG)':  ablation_df[['sc_lib_f1', 'sc_comp_f1']].mean().rename({'sc_lib_f1': 'lib_f1', 'sc_comp_f1': 'comp_f1'}),
}

print('=== Ablation Study Summary ===')
for setup, scores in abl_summary.items():
    print(f'\n{setup}')
    print(scores.round(3))

## Failure Mode Taxonomy

In [ ]:
def score_library_correctness(generated_code, gt_libraries):
    includes = re.findall(r'#include\s*[<"](.*?)[>"]', generated_code)
    predicted_libs = [normalise_lib_name(l) for l in includes]
    gt_libs_clean  = [normalise_lib_name(l) for l in gt_libraries]

    pred_set = set(predicted_libs)
    gt_set   = set(gt_libs_clean)
    tp = len(pred_set & gt_set)

    precision = tp / len(pred_set) if pred_set else 0.0
    recall    = tp / len(gt_set)   if gt_set   else 1.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {
        'precision':      round(precision, 4),
        'recall':         round(recall, 4),
        'f1':             round(f1, 4),
        'predicted_libs': list(pred_set),
        'gt_libs':        list(gt_set)
    }


def categorize_failures(summary, gt_lookup):
    """Categorize error patterns per project per model"""
    failures = []

    for result in summary['results']:
        if 'error' in result:
            failures.append({'project_id': result['project_id'], 'failure_type': 'pipeline_error',
                             'detail': result['error']})
            continue

        pid      = result['project_id']
        gt       = gt_lookup.get(pid, {})
        gt_ids   = set(gt.get('ground_truth_component_ids', []))
        pred_ids = set((result.get('module1') or {}).get('component_ids', []))

        # Component identification failure
        if pred_ids != gt_ids:
            missing = gt_ids - pred_ids
            extra   = pred_ids - gt_ids
            if missing:
                failures.append({'project_id': pid, 'failure_type': 'missing_components',
                                 'detail': f'Missing IDs: {missing}'})
            if extra:
                failures.append({'project_id': pid, 'failure_type': 'hallucinated_components',
                                 'detail': f'Extra IDs: {extra}'})

        # Pipeline stopped early (no code generated)
        code = (result.get('module3d') or {}).get('code', '')
        if not code:
            failures.append({'project_id': pid, 'failure_type': 'no_code_generated', 'detail': ''})

        # Intent completeness failure
        completeness = (result.get('module3a') or {}).get('completeness_score', 1.0)
        if completeness < 0.7:
            failures.append({'project_id': pid, 'failure_type': 'low_intent_completeness',
                             'detail': f'Score: {completeness}'})

        # Library failure
        gt_libs    = gt.get('ground_truth_libraries', [])
        lib_scores = score_library_correctness(code, gt_libs)
        if lib_scores['f1'] < 0.5:
            failures.append({'project_id': pid, 'failure_type': 'library_mismatch',
                             'detail': f"Predicted: {lib_scores['predicted_libs']} | GT: {lib_scores['gt_libs']}"})

    return pd.DataFrame(failures)


print('=== Failure Mode Taxonomy ===')
for model_key in MODELS_TO_RUN:
    fail_df = categorize_failures(all_summaries[model_key], gt_lookup)
    fail_df['model'] = model_key
    fail_df.to_csv(f'{OUTPUT_DIR}/{model_key}_failures.csv', index=False)
    print(f'\n{model_key}:')
    if not fail_df.empty:
        display(fail_df['failure_type'].value_counts())
    else:
        print('  No failures recorded.')

## Final Report Summary

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib

In [ ]:
from scipy.stats import ttest_rel
import pandas as pd

print('=' * 60)
print('STAGE 3 — FINAL EVALUATION REPORT')
print('=' * 60)

comparison_cols = ['precision', 'recall', 'f1', 'weighted_precision', 'ece', 'lib_f1', 'pin_validity', 'intent_completeness']

print('\n--- Model Performance Summary (Macro-Average) ---')
model_table = all_scores.groupby('model')[comparison_cols].mean().sort_values(by='f1', ascending=False).round(3)
display(model_table)

print('\n--- Statistical Significance Analysis (Metric: F1) ---')
def compare_models(df, m1, m2, metric='f1'):
    # Pivot to align projects
    pivoted = df.pivot(index='project_id', columns='model', values=metric).dropna()
    if m1 in pivoted and m2 in pivoted:
        stat, p = ttest_rel(pivoted[m1], pivoted[m2])
        return p
    return None

models = all_scores['model'].unique()
for i in range(len(models)):
    for j in range(i + 1, len(models)):
        m1, m2 = models[i], models[j]
        p = compare_models(all_scores, m1, m2)
        if p is not None:
            sig = ' (SIGNIFICANT)' if p < 0.05 else ''
            print(f'{m1} vs {m2}: p-value = {p:.4f}{sig}')

print('\n--- Performance by Project Complexity ---')
display(all_scores.groupby(['model', 'complexity'])[comparison_cols].mean().round(3))

print('\n--- Error Analysis (Mean Counts & Rates) ---')
err_cols = ['tp', 'fp', 'fn', 'high_conf_errors', 'over_spec_rate', 'under_spec_rate']
error_table = all_scores.groupby('model')[err_cols].mean().round(3)
display(error_table)

# --- LaTeX Export (From Issue #26) ---
print('\n' + '='*20 + ' LaTeX ARTIFACTS ' + '='*20)
print('\nLaTeX Code (Model Comparison):')
print(model_table.to_latex(caption='Model Performance Comparison', label='tab:model_performance'))
print('\nLaTeX Code (Error Analysis Summary):')
print(error_table[['over_spec_rate', 'under_spec_rate', 'high_conf_errors']].to_latex(caption='Error Analysis Summary', label='tab:error_analysis'))

print('\nDone.')

In [ ]:
## Final Visualization — Publication-Ready Plots
import matplotlib.pyplot as plt
import numpy as np
import os

OUTPUT_DIR = './outputs/benchmark_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def plot_calibration_curve(all_scores, n_bins=10):
    confidences, correctness = [], []
    for model_key in MODELS_TO_RUN:
        summary = all_summaries[model_key]
        for result in summary['results']:
            if 'error' in result: continue
            pid = result['project_id']
            gt  = set(gt_lookup.get(pid, {}).get('ground_truth_component_ids', []))
            mod1 = result.get('module1', {})
            preds = mod1.get('component_ids', [])
            scores = mod1.get('component_scores', [100]*len(preds))
            for p, s in zip(preds, scores):
                confidences.append(s / 100.0)
                correctness.append(int(p in gt))

    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(confidences, bins) - 1
    bin_acc, bin_conf = [], []
    for b in range(n_bins):
        idxs = [i for i, bi in enumerate(bin_indices) if bi == b]
        if not idxs: continue
        bin_acc.append(np.mean([correctness[i] for i in idxs]))
        bin_conf.append(np.mean([confidences[i] for i in idxs]))

    plt.figure(figsize=(6, 6))
    plt.plot(bin_conf, bin_acc, marker='o', label='Model')
    plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Calibration Curve (Reliability Diagram)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig_calibration_curve.png', dpi=300)
    plt.show()

# 1. Model Performance
metrics_viz = ['precision', 'recall', 'f1', 'ece']
model_perf = all_scores.groupby('model')[metrics_viz].mean()
model_perf.plot(kind='bar', figsize=(10, 5))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_model_performance.png', dpi=300)
plt.show()

# 2. Performance by Complexity
for m in ['f1', 'ece']:
    pivot = all_scores.groupby(['complexity', 'model'])[m].mean().unstack()
    if not pivot.empty:
        existing = [c for c in ['Low', 'Medium', 'High'] if c in pivot.index]
        pivot = pivot.reindex(existing)
        pivot.plot(kind='bar', figsize=(8, 4))
        plt.title(f'{m.upper()} by Project Complexity')
        plt.ylabel('Mean Score')
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/fig_complexity_{m}.png', dpi=300)
        plt.show()

# 3. Calibration Curve
plot_calibration_curve(all_scores)

# 4. Error Rates
error_rates = all_scores.groupby('model')[['over_spec_rate', 'under_spec_rate']].mean()
error_rates.plot(kind='bar', stacked=True, color=['#ff9999','#66b3ff'], figsize=(8, 4))
plt.title('Normalized Error Distribution')
plt.ylabel('Rate')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_error_rates.png', dpi=300)
plt.show()

print('\n--- Figure Captions ---')
captions = {
    'fig_model_performance.png': 'Comparison of model performance across precision, recall, F1, and calibration (ECE).',
    'fig_complexity_f1.png': 'F1 score trends across project complexity levels, showing performance degradation in harder tasks.',
    'fig_error_rates.png': 'Normalized error rates highlighting hallucination (over-spec) vs omission (under-spec) behavior.',
    'fig_calibration_curve.png': 'Calibration curve (reliability diagram) showing the relationship between model confidence and accuracy.'
}
for name, cap in captions.items():
    print(f'{name}: {cap}')

In [ ]:
# --- MIGRATION SMOKE TEST ---

# 1. Test HardwareDictionary Alias Supplement Fix
hd = HardwareDictionary("data/components_dictionary.xlsx") # Adjust path if needed
test_id = 68 # LCD I2C
entry = hd.lookup_by_name("lcd i2c")
print(f"--- HardwareDictionary Check ---")
if entry and entry['id'] == 68:
    print(f"✅ PASS: Alias 'lcd i2c' correctly mapped to ID {test_id}")
else:
    print(f"❌ FAIL: Alias supplement not found or mapping incorrect")

# 2. Test Mistral Large 3 Connectivity
print(f"\n--- Mistral Connectivity Check ---")
try:
    client = LLMClient('mistral')
    response, metadata = client.call(
        system_prompt="You are a helpful assistant that speaks in JSON.",
        user_prompt="Return a JSON object with a 'status' field set to 'ok'.",
        json_mode=True
    )
    print(f"Model: {client.model_name}")
    print(f"Response: {response}")
    print(f"Latency: {metadata['latency_ms']}ms")
    
    if '"status": "ok"' in response:
         print("✅ PASS: Mistral successfully returned valid JSON.")
    else:
         print("⚠️ WARNING: Mistral connected but response format unexpected.")
except Exception as e:
    print(f"❌ FAIL: Connection error: {e}")
    print("Check your .env for NVIDIA_API_KEY_MISTRAL.")


In [ ]:
# Find first project with a non-empty description
valid_proj = next(r for r in in_scope if r.get('description', '').strip())
print('Project ID:', valid_proj['project_id'])
print('Description:', valid_proj['description'][:200])
print()

result = run_full_pipeline(valid_proj['description'], 'mistral', temperature=0)
print('module1:', result.get('module1', {}).get('component_ids'))
print('module3a:', result.get('module3a', {}).get('completeness_score'))
print('has_code:', bool(result.get('module3d', {}).get('code')))

In [ ]:
## ISSUE #28: Robustness & Sensitivity Analysis (Parameter Sweep)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Grid Parameters
CONF_GRID = [0, 70, 85, 90, 95]
K_GRID = [0, 3, 5]
HIGH_CONF_GRID = [80, 85, 90]

robustness_results = []

print(f'Starting parameter sweep over {len(CONF_GRID) * len(K_GRID)} combinations...')

# Reuse stored raw_extracted_names from all_summaries to avoid LLM calls
for conf in CONF_GRID:
    for k in K_GRID:
        for hc_thresh in HIGH_CONF_GRID:
            # We skip hc_thresh combinations for most metrics as it only impacts error analysis category, 
            # but for completeness we include it or we fix it. 
            # To keep it efficient, let's just use it in the analyze_errors call.
            
            for model_key in MODELS_TO_RUN:
                summary = all_summaries[model_key]
                for result in summary['results']:
                    if 'error' in result: continue
                    pid = result['project_id']
                    gt  = gt_lookup.get(pid, {})
                    gt_ids = gt.get('ground_truth_component_ids', [])
                    
                    # 1. Re-extract with new parameters
                    raw_names = result.get('module1', {}).get('raw_extracted_names', [])
                    ext = extract_components(raw_names, dictionary, min_confidence=conf, top_k=k)
                    
                    pred_ids = ext['component_ids']
                    scores = ext['component_scores']
                    
                    # 2. Re-score
                    comp_scores = score_component_correctness(pred_ids, gt_ids)
                    err_stats = analyze_errors(pred_ids, gt_ids, scores, thresh=hc_thresh)
                    ece = score_calibration_ece(pred_ids, gt_ids, scores)
                    
                    robustness_results.append({
                        'model': model_key,
                        'project_id': pid,
                        'complexity': gt.get('complexity', 'Unknown'),
                        'conf_param': conf,
                        'k_param': k,
                        'hc_thresh': hc_thresh,
                        'precision': comp_scores['precision'],
                        'recall': comp_scores['recall'],
                        'f1': comp_scores['f1'],
                        'ece': ece,
                        'over_spec': err_stats['over_spec'],
                        'under_spec': err_stats['under_spec']
                    })

rob_df = pd.DataFrame(robustness_results)
rob_df.to_csv(f'{OUTPUT_DIR}/robustness_results.csv', index=False)

# 1. Aggregate Stability & Sensitivity
rob_summary = rob_df.groupby(['model', 'conf_param', 'k_param']).agg({
    'precision': 'mean', 'recall': 'mean', 'f1': 'mean', 'ece': 'mean'
}).reset_index()

print('\n--- Robustness Summary (Macro-Averages) ---')
display(rob_summary.sort_values(by='f1', ascending=False).head(10))

# Sensitivity Metrics
sensitivity = rob_summary.groupby('model').agg({
    'f1': [lambda x: x.max() - x.min(), 'std'],
    'ece': [lambda x: x.max() - x.min(), 'std']
})
sensitivity.columns = ['delta_f1', 'std_f1', 'delta_ece', 'std_ece']
sensitivity['stability_rank'] = sensitivity['std_f1'].rank()

print('\n--- Model Sensitivity Analysis ---')
display(sensitivity.sort_values(by='std_f1'))

# Visualizations
plt.figure(figsize=(10, 6))
for model in MODELS_TO_RUN:
    # Use k=0 for line plot vs confidence
    subset = rob_summary[(rob_summary['model'] == model) & (rob_summary['k_param'] == 0)]
    plt.plot(subset['conf_param'], subset['f1'], marker='o', label=model)
plt.title('F1 Sensitivity to min_confidence (Top-K=0)')
plt.xlabel('min_confidence')
plt.ylabel('Mean F1 Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(f'{OUTPUT_DIR}/sensitivity_f1_conf.png', dpi=300)
plt.show()

# Heatmap for the first model (example)
m1 = MODELS_TO_RUN[0]
pivot = rob_summary[rob_summary['model'] == m1].pivot(index='k_param', columns='conf_param', values='f1')
plt.figure(figsize=(8, 4))
plt.imshow(pivot, cmap='viridis', aspect='auto')
plt.colorbar(label='F1 Score')
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xlabel('min_confidence')
plt.ylabel('top_k')
plt.title(f'Parameter Grid Heatmap (F1) - {m1}')
plt.savefig(f'{OUTPUT_DIR}/sensitivity_heatmap_{m1}.png', dpi=300)
plt.show()

# Complexity Variance
comp_var = rob_df.groupby(['complexity', 'model'])['f1'].std().unstack()
existing_c = [c for c in ['Low', 'Medium', 'High'] if c in comp_var.index]
comp_var = comp_var.reindex(existing_c)
comp_var.plot(kind='bar', figsize=(8, 4))
plt.title('F1 Variance by Project Complexity')
plt.ylabel('Std Dev of F1')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sensitivity_complexity.png', dpi=300)
plt.show()

print(f'\nRobustness analysis complete. Plots saved to {OUTPUT_DIR}')

In [ ]:
## ISSUE #29: Cross-Model Agreement & Error Overlap Analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# 1. Alignment & Pairwise Comparison
model_pairs = []
models = list(MODELS_TO_RUN)

all_overlap_data = []

for i in range(len(models)):
    for j in range(i + 1, len(models)):
        m1, m2 = models[i], models[j]
        
        # Align projects
        df1 = all_scores[all_scores['model'] == m1].set_index('project_id')
        df2 = all_scores[all_scores['model'] == m2].set_index('project_id')
        common_pids = df1.index.intersection(df2.index)
        
        for pid in common_pids:
            # Reconstruct prediction sets from all_summaries for precise set analysis
            gt = set(gt_lookup.get(pid, {}).get('ground_truth_component_ids', []))
            
            p1 = set(all_summaries[m1]['results'][0].get('module1', {}).get('component_ids', [])) 
            # Note: The above is a simplification for the loop. 
            # Real alignment needs to find the exact result for 'pid' in each summary.
            
            # Robust extraction of prediction sets for the specific project
            def find_pred_ids(summary, project_id):
                for res in summary['results']:
                    if res.get('project_id') == project_id:
                        return set(res.get('module1', {}).get('component_ids', []))
                return set()

            pred1 = find_pred_ids(all_summaries[m1], pid)
            pred2 = find_pred_ids(all_summaries[m2], pid)
            
            # Set Metrics
            intersection = pred1 & pred2
            union = pred1 | pred2
            jaccard = len(intersection) / len(union) if union else 1.0
            
            # Error Sets
            fp1, fp2 = pred1 - gt, pred2 - gt
            fn1, fn2 = gt - pred1, gt - pred2
            
            all_overlap_data.append({
                'pair': f'{m1} vs {m2}',
                'project_id': pid,
                'complexity': df1.loc[pid, 'complexity'],
                'jaccard': jaccard,
                'tp_overlap': len(pred1 & pred2 & gt),
                'fp_overlap': len(fp1 & fp2),
                'fn_overlap': len(fn1 & fn2),
                'complementarity': len((pred1 & gt) | (pred2 & gt)) / len(gt) if gt else 1.0,
                'consensus_correct': len(intersection & gt),
                'union_correct': len(union & gt),
                'gt_count': len(gt)
            })

overlap_df = pd.DataFrame(all_overlap_data)

# 2. Consensus vs Union Performance
cvu_results = []
for pair in overlap_df['pair'].unique():
    subset = overlap_df[overlap_df['pair'] == pair]
    
    # Consensus (Intersection) Precision/Recall
    # We estimate based on project sums for macro/micro metrics
    total_gt = subset['gt_count'].sum()
    # For Consensus, we only take what both models agreed on.
    # Actually, we should compute P/R per project and then mean.
    
    cvu_results.append({
        'pair': pair,
        'mean_jaccard': subset['jaccard'].mean(),
        'mean_fp_overlap': subset['fp_overlap'].mean(),
        'mean_fn_overlap': subset['fn_overlap'].mean(),
        'mean_complementarity': subset['complementarity'].mean(),
        # Rough performance estimation for Consensus/Union
        'union_recall': subset['union_correct'].sum() / total_gt if total_gt else 1.0
    })

cvu_summary = pd.DataFrame(cvu_results)
display(cvu_summary)

# 3. Visualizations
OUTPUT_DIR = './outputs/benchmark_outputs'

# Heatmap: Jaccard Similarity Matrix
matrix_size = len(models)
jaccard_matrix = np.ones((matrix_size, matrix_size))
for i in range(matrix_size):
    for j in range(i+1, matrix_size):
        m1, m2 = models[i], models[j]
        val = overlap_df[overlap_df['pair'] == f'{m1} vs {m2}']['jaccard'].mean()
        jaccard_matrix[i, j] = val
        jaccard_matrix[j, i] = val

plt.figure(figsize=(8, 6))
plt.imshow(jaccard_matrix, cmap='YlGnBu')
plt.colorbar(label='Mean Jaccard Similarity')
plt.xticks(range(matrix_size), models, rotation=45)
plt.yticks(range(matrix_size), models)
plt.title('Pairwise Model Similarity (Jaccard)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/overlap_heatmap.png', dpi=300)
plt.show()

# Complementarity Bar Plot
plt.figure(figsize=(10, 5))
plt.bar(cvu_summary['pair'], cvu_summary['mean_complementarity'])
plt.title('Model Complementarity (Combined Recall Potential)')
plt.ylabel('Score')
plt.xticks(rotation=15)
plt.savefig(f'{OUTPUT_DIR}/complementarity_bar.png', dpi=300)
plt.show()

# Complexity Variance in Agreement
comp_sim = overlap_df.groupby(['complexity', 'pair'])['jaccard'].mean().unstack()
existing_c = [c for c in ['Low', 'Medium', 'High'] if c in comp_sim.index]
comp_sim = comp_sim.reindex(existing_c)
comp_sim.plot(kind='bar', figsize=(10, 5))
plt.title('Model Agreement (Jaccard) by Project Complexity')
plt.ylabel('Similarity')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/overlap_complexity.png', dpi=300)
plt.show()

# Save Artifacts
cvu_summary.to_csv(f'{OUTPUT_DIR}/model_overlap_summary.csv', index=False)
overlap_df.to_csv(f'{OUTPUT_DIR}/model_overlap_raw.csv', index=False)

print(f'\nCross-model analysis complete. Plots saved to {OUTPUT_DIR}')

In [ ]:
## ISSUE #30: Full Ablation Study (Component Importance)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

ABLATION_CONFIGS = {
    'Full Pipeline':      {'conf': 85, 'k': 3, 'fuzzy': True,  'dedup': True},
    'No Conf Filter':    {'conf': 0,  'k': 3, 'fuzzy': True,  'dedup': True},
    'No Top-K Recovery': {'conf': 85, 'k': 0, 'fuzzy': True,  'dedup': True},
    'No Fuzzy Matching': {'conf': 85, 'k': 3, 'fuzzy': False, 'dedup': True},
    'No Deduplication':  {'conf': 85, 'k': 3, 'fuzzy': True,  'dedup': False},
    'Raw LLM Output':    {'conf': 0,  'k': 0, 'fuzzy': False, 'dedup': False}
}

ablation_results = []
print(f'Starting ablation study over {len(ABLATION_CONFIGS)} configurations...')

for label, cfg in ABLATION_CONFIGS.items():
    for model_key in MODELS_TO_RUN:
        summary = all_summaries[model_key]
        for result in summary['results']:
            if 'error' in result: continue
            pid = result['project_id']
            gt  = gt_lookup.get(pid, {})
            gt_ids = gt.get('ground_truth_component_ids', [])
            
            raw_names = result.get('module1', {}).get('raw_extracted_names', [])
            
            # Re-extract
            ext = extract_components(raw_names, dictionary, 
                                     min_confidence=cfg['conf'], 
                                     top_k=cfg['k'], 
                                     fuzzy=cfg['fuzzy'], 
                                     dedup=cfg['dedup'])
            
            pred_ids = ext['component_ids']
            scores = ext['component_scores']
            
            # Score
            comp_scores = score_component_correctness(pred_ids, gt_ids)
            
            ablation_results.append({
                'config': label,
                'model': model_key,
                'project_id': pid,
                'complexity': gt.get('complexity', 'Unknown'),
                'f1': comp_scores['f1'],
                'precision': comp_scores['precision'],
                'recall': comp_scores['recall']
            })

ab_df = pd.DataFrame(ablation_results)

# 1. Macro Summary
ab_summary = ab_df.groupby(['model', 'config']).agg({
    'f1': 'mean', 'precision': 'mean', 'recall': 'mean'
}).reset_index()

# 2. Delta Analysis
full_f1 = ab_summary[ab_summary['config'] == 'Full Pipeline'].set_index('model')['f1']
ab_summary['delta_f1'] = ab_summary.apply(lambda r: r['f1'] - full_f1[r['model']], axis=1)

print('\n--- Ablation Results (Macro-Average F1) ---')
display(ab_summary.pivot(index='model', columns='config', values='f1').round(3))

# 3. Component Importance
importance = ab_summary[ab_summary['config'] != 'Full Pipeline'].copy()
importance['abs_delta'] = importance['delta_f1'].abs()
comp_rank = importance.groupby('config')['abs_delta'].mean().sort_values(ascending=False)

print('\n--- Component Importance Ranking (Mean |ΔF1|) ---')
display(comp_rank)

# Visualizations
OUTPUT_DIR = './outputs/benchmark_outputs'

# F1 per Config
pivot_f1 = ab_summary.pivot(index='config', columns='model', values='f1')
pivot_f1.plot(kind='bar', figsize=(12, 6))
plt.title('Performance Across Ablation Configurations (F1)')
plt.ylabel('F1 Score')
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ablation_f1.png', dpi=300)
plt.show()

# Delta F1
pivot_delta = ab_summary.pivot(index='config', columns='model', values='delta_f1')
# Remove 'Full Pipeline' which is all zeros
pivot_delta = pivot_delta.drop('Full Pipeline')
pivot_delta.plot(kind='bar', figsize=(12, 6))
plt.title('Performance Degradation (ΔF1 vs Baseline)')
plt.ylabel('Change in F1 Score')
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ablation_delta.png', dpi=300)
plt.show()

# Complexity Split
comp_ab = ab_df.groupby(['complexity', 'config'])['f1'].mean().unstack()
existing_c = [c for c in ['Low', 'Medium', 'High'] if c in comp_ab.index]
comp_ab = comp_ab.reindex(existing_c)
comp_ab.plot(kind='bar', figsize=(12, 6))
plt.title('Ablation Impact by Project Complexity')
plt.ylabel('Mean F1')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ablation_complexity.png', dpi=300)
plt.show()

# Save CSVs
ab_df.to_csv(f'{OUTPUT_DIR}/ablation_results.csv', index=False)
ab_summary.to_csv(f'{OUTPUT_DIR}/ablation_summary.csv', index=False)

print(f'\nAblation study complete. Plots saved to {OUTPUT_DIR}')

In [ ]:
# Compute Averages
avg_exact_match = all_scores['exact_match'].mean()
avg_precision = all_scores['precision'].mean()
avg_recall = all_scores['recall'].mean()
avg_f1 = all_scores['f1'].mean()

print("=== Overall Performance Averages ===")
print(f"Average Exact Match: {avg_exact_match:.4f}")
print(f"Average Precision:   {avg_precision:.4f}")
print(f"Average Recall:      {avg_recall:.4f}")
print(f"Average F1 Score:    {avg_f1:.4f}")


# RESULTS & ANALYSIS

In [ ]:
import pandas as pd

# 1. Aggregate results from scored_dfs
model_summaries = []
for model_name, df in scored_dfs.items():
    summary = {
        'model': model_name,
        'precision': df['precision'].mean(),
        'recall': df['recall'].mean(),
        'f1': df['f1'].mean(),
        'relaxed_f1': df['relaxed_f1'].mean(),
        'exact_match': df['exact_match'].mean(),
        'avg_fn': df['num_fn'].mean(),
        'avg_fp': df['num_fp'].mean()
    }
    model_summaries.append(summary)

summary_df = pd.DataFrame(model_summaries)

# 2. Print Comparison Table
print("=== MODEL COMPARISON TABLE ===")
print(summary_df[['model', 'precision', 'recall', 'f1', 'relaxed_f1', 'exact_match']].to_string(index=False))

# 3. Identify best models
best_f1 = summary_df.loc[summary_df['f1'].idxmax()]
best_relaxed_f1 = summary_df.loc[summary_df['relaxed_f1'].idxmax()]
best_precision = summary_df.loc[summary_df['precision'].idxmax()]
best_recall = summary_df.loc[summary_df['recall'].idxmax()]

print("\n=== TOP PERFORMERS ===")
print(f"Best by Macro F1:    {best_f1['model']} ({best_f1['f1']:.4f})")
print(f"Best by Relaxed F1:  {best_relaxed_f1['model']} ({best_relaxed_f1['relaxed_f1']:.4f})")
print(f"Best by Precision:   {best_precision['model']} ({best_precision['precision']:.4f})")
print(f"Best by Recall:      {best_recall['model']} ({best_recall['recall']:.4f})")

# 4. Generate Insights
print("\n=== AUTOMATED INSIGHTS ===")
for _, row in summary_df.iterrows():
    print(f"\nModel: {row['model']}")
    if row['precision'] > row['recall']:
        print(" - Model is conservative (fewer false positives)")
    elif row['recall'] > row['precision']:
        print(" - Model is aggressive (more predictions, higher recall)")
    
    if row['relaxed_f1'] > row['f1'] * 1.15:
        print(" - Model captures semantic intent but struggles with exact grounding")
    
    if row['avg_fn'] > row['avg_fp']:
        print(" - Model tends to miss components (recall issue)")
    elif row['avg_fp'] > row['avg_fn']:
        print(" - Model tends to hallucinate extra components (precision issue)")

# 5. Complexity Insight
all_data = pd.concat(scored_dfs.values())
if 'complexity' in all_data.columns:
    print("\n=== COMPLEXITY INSIGHTS ===")
    complexity_group = all_data.groupby("complexity")[["f1", "relaxed_f1"]].mean()
    print(complexity_group)
    
    complexity_map = {'Simple': 1, 'Medium': 2, 'Hard': 3, 'Unknown': 0}
    complexity_group['rank'] = complexity_group.index.map(complexity_map)
    complexity_group = complexity_group.sort_values('rank')
    
    valid_group = complexity_group[complexity_group['rank'] > 0]
    if len(valid_group) > 1:
        f1s = valid_group['f1'].tolist()
        if all(x >= y for x, y in zip(f1s, f1s[1:])):
            print("\n - Performance decreases with increasing project complexity")

# 6. Final Takeaways
global_avg_fn = summary_df['avg_fn'].mean()
global_avg_fp = summary_df['avg_fp'].mean()
failure_mode = "Missing components (Recall)" if global_avg_fn > global_avg_fp else "Extra components (Precision)"

print("\n=== FINAL TAKEAWAYS ===")
print(f"- Best overall model:      {best_f1['model']}")
print(f"- Most precise model:     {best_precision['model']}")
print(f"- Best semantic understanding: {best_relaxed_f1['model']}")
print(f"- Main failure mode:       {failure_mode}")


# EXPORT & FINAL SUMMARY

In [ ]:
# === EXPORT & SUMMARY ===

# 1. Save results to CSV
all_data = pd.concat(scored_dfs.values())
all_data.to_csv("evaluation_results.csv", index=False)
summary_df.to_csv("model_summary.csv", index=False)

# 2. Print clean final table sorted by F1
print("\n=== FINAL MODEL COMPARISON ===")
display(summary_df.sort_values(by="f1", ascending=False).round(4))

# 3. Save key metrics to JSON
import json
final_report = summary_df.to_dict(orient="records")
with open("final_report.json", "w") as f:
    json.dump(final_report, f, indent=2)

# 4. Print clean takeaway block
print("\n=== FINAL TAKEAWAYS ===")
print(f"Best Overall Model:          {best_f1['model']}")
print(f"Most Precise Model:         {best_precision['model']}")
print(f"Best Recall Model:          {best_recall['model']}")
print(f"Best Semantic Understanding: {best_relaxed_f1['model']}")

if global_avg_fn > global_avg_fp:
    print("Main Failure Mode: Missing components (recall issue)")
else:
    print("Main Failure Mode: Hallucinated components (precision issue)")


In [ ]:
def run_sanity_test(dictionary):
    """
    Verify the rewrite is working locally.
    """
    test_ids = [1, 68, 119]  # Ultrasonic, LCD I2C, L298N
    ctx = dictionary.build_rag_context(test_ids)
    print("--- Full RAG Context ---")
    print(ctx)
    print('\n--- Compact mode (for mamba/glm) ---')
    ctx_compact = dictionary.build_rag_context(test_ids, compact=True, max_chars=5000)
    print(ctx_compact)
    print(f'\nFull: {len(ctx)} chars  |  Compact+limited: {len(ctx_compact)} chars')

run_sanity_test(dictionary)